In [ ]:
import json
import matplotlib.pyplot as plt
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
import pandas as pd
from scipy.signal import hilbert
from scipy.io import loadmat, savemat

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

In [ ]:
bands={i:b for i, b in enumerate([[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]], start=1)}

f_sample = 1000
def fs_band(band):
    return int(bands[band][1]*2*1.125+0.5)
fixTime = [[np.empty((1120//i, 195, 50)) for __ in bands] for i in [1,2,4]]
fixSamples = [[np.empty((280, 195, 50)) for __ in bands] for i in [1,2,4]]
BLP = []
zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
samp_per_sub = {}
for sub in range(50):
    archive = h5py.File(io.BytesIO(zip_file.read(f"Sub_{sub+1}/electrode_BLP.mat")))

    for band in tqdm(bands, desc="Band", total=9):
        band_data = archive[archive["electrode_BLP"][0,band-1]]
        for i in range(band_data.shape[0]):
            this_piece = archive[band_data[i,0]]
            BLP.append(this_piece)
            break
        break
    break

In [ ]:
bands={i:b for i, b in enumerate([[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]], start=1)}

f_sample = 1000
def fs_band(band):
    return int(bands[band][1]*2*1.125+0.5)
fixTime = [np.empty((45*fs_band(band), 195, 50)) for band in bands]
fixSamples = [np.empty((1000, 195, 50)) for __ in bands]

zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
for subject in tqdm(range(50), desc="Subject"):
    archive = h5py.File(io.BytesIO(zip_file.read(f"Sub_{subject+1}/EEG_bands.mat")))

    for band in tqdm(bands, desc="Band", total=9):
        band_data = archive[archive["EEG_bands"][0,band-1]]
        VOLT = []

        for i in range(band_data.shape[0]):
            this_piece = archive[band_data[i,0]]
            VOLT.append(this_piece)
            break
        break
    break


In [ ]:
VOLT[0].shape[0],BLP[0].shape[0]*125

In [ ]:
z = hilbert(VOLT[0][:,0])
power = np.absolute(z)
dsp = np.average(np.reshape(power[:8500],(-1,125)),1)

In [ ]:
np.average(power[dsp.shape[0]*125:]), BLP[0][-1,0]

In [ ]:
plt.plot(np.arange(BLP[0].shape[0])/8,BLP[0][:,0], lw=3, label="your BLP")
plt.plot(np.arange(VOLT[0].shape[0])/1000,VOLT[0][:,0], label="your voltage")
plt.plot(np.arange(VOLT[0].shape[0])/1000,power, lw=1, label="my BLP")
plt.plot(np.arange(dsp.shape[0])/8,dsp, ":", label="my BLP\naverage\n125 samples")
plt.title(r"# samples voltage $\approx 125 \times$ # samples BLP")
plt.legend()
plt.show()

# Select most non-linear subject

In [ ]:
selected_subjects = {}
results_path = "../NonLinearityResultsNew/"
results_folder = "NEW_electrodeBLP_fixedTime_avg1_band{}_bin10"

for band in range(1,10):
    this_folder = results_folder.format(band)
    results_file = this_folder+"_globalStats.json"
    with open(os.path.join(results_path,this_folder,results_file)) as fp:
        globalStats = {k:np.array(v) for k,v in json.load(fp).items()}
        relMI = 1 - globalStats['gaussMI'] / globalStats['totalMI']
        selected_subjects[band] = np.argmax(relMI)
with open("selected_subjects.json", "w") as fp:
    json.dump(selected_subjects, fp, cls=NpEncoder)


# Find most non-linear pair per correlation bin

In [ ]:
bins = np.arange(21)/10-1
selected_pairs = {}
selected_linear = {}
for band in range(1,10):
    selected_pairs[band] = {}
    selected_linear[band] = {}
    this_folder = results_folder.format(band)
    subj = selected_subjects[band]
    corrector = Corrector(10,1120,200, 50000,1120,os.path.join(results_path,this_folder))
    corrector.compute_correction()
    corr = np.load(os.path.join(results_path,this_folder,f"subject{subj:02}_10_cor.npy"))
    binned_corr = pd.cut(corr, bins, labels=False)
    MI = corrector.correct(np.load(os.path.join(results_path,this_folder,f"subject{subj:02}_10.npy")))
    tMI = MI[:,0]
    gMI = np.average(MI[:,1:], 1)
    rMI = 1 - gMI / tMI
    rMI[rMI==np.inf] = -np.inf
    for bin in range(np.min(binned_corr), np.max(binned_corr)+1):
        tmp_rMI = rMI.copy()
        tmp_rMI[binned_corr != bin] = -np.inf
        max_pair = np.argmax(tmp_rMI)
        selected_pairs[band][bin] = [np.triu_indices(195, 1)[0][max_pair],np.triu_indices(195, 1)[1][max_pair]]
        selected_linear[band][bin] = max_pair
with open("selected_pairs.json", "w") as fp:
    json.dump(selected_pairs, fp, cls=NpEncoder)
with open("selected_linear.json", "w") as fp:
    json.dump(selected_linear, fp, cls=NpEncoder)

In [ ]:
interesting_electrodes = {}
for band in selected_pairs:
    electrodes = set()
    for bin in selected_pairs[band]:
        for e in selected_pairs[band][bin]:
            electrodes.add(e)
    interesting_electrodes[band] = sorted(electrodes)

with open("interesting_electrodes.json", "w") as fp:
    json.dump(interesting_electrodes, fp, cls=NpEncoder)

# Generate surrogates

In [ ]:
zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")

for band in tqdm(range(1,10), desc="Band", total=9):
    subject = selected_subjects[band]
    archive = h5py.File(io.BytesIO(zip_file.read(f"Sub_{subject+1}/EEG_bands.mat")))
    band_data = archive[archive["EEG_bands"][0,band-1]]

    surrogates = []
    for s in tqdm(range(99)):
        tmp_pieces = []
        tot_len = 0
        for i in range(band_data.shape[0]):
            this_piece = archive[band_data[i,0]][:,interesting_electrodes[band]]
            tmp = ms.surrogate(this_piece)
            z = hilbert(tmp, axis=0)
            power = np.absolute(z)
            full_samp = power.shape[0]//125
            extra = full_samp*125 < power.shape[0]
            dsp = np.zeros((full_samp+extra, power.shape[1]))
            dsp[:full_samp,:] = np.average(np.reshape(power[:full_samp*125],(-1,125,power.shape[1])),1)
            if extra:
                dsp[-1,:] = np.average(power[full_samp*125:,:],0)
            tmp_pieces.append(dsp.copy())
            tot_len += dsp.shape[0]
            if tot_len > 1120:
                break
        surrogates.append(np.concatenate(tmp_pieces)[:1120,:])
    new_band = np.moveaxis(np.array(surrogates),0,2)
    savemat(f"../NonLinearityResultsNew/NEW_electrodeBLP_fixedTime_avg1_surrogates/surrogateBLP_band{band}.mat", {"BLP":new_band})

In [ ]:
with open("selected_subjects.json") as fp:
    selected_subjects = {int(k):v for k, v in json.load(fp).items()}
with open("selected_pairs.json") as fp:
    selected_pairs = {int(k):{int(k2):v2 for k2, v2 in v.items()} for k, v in json.load(fp).items()}
with open("selected_linear.json") as fp:
    selected_linear = {int(k):{int(k2):v2 for k2, v2 in v.items()} for k, v in json.load(fp).items()}
with open("interesting_electrodes.json") as fp:
    interesting_electrodes = {int(k):v for k, v in json.load(fp).items()}

In [ ]:
band_names= [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
nle = NonLinearEstimator(bins=10, surrogates=0, cache="cache", workers=20, verbose=True)
for band in range(1,10):
    this_folder = results_folder.format(band)
    subj = selected_subjects[band]
    corrector = Corrector(10,1120,200, 50000,1120,os.path.join(results_path,this_folder))
    corrector.compute_correction()
    MI = corrector.correct(np.load(os.path.join(results_path,this_folder,f"subject{subj:02}_10.npy")))
    data = loadmat(f"../NonLinearityResultsNew/NEW_electrodeBLP_fixedTime_avg1_surrogates/surrogateBLP_band{band}.mat")["BLP"]
    VSMI = corrector.correct(nle.estimate(data, extended_stats=False, save_out=False, display=False)[1])
    sel_el_num = len(interesting_electrodes[band])
    for bin in selected_pairs[band]:
        old_surr = MI[selected_linear[band][bin], 1:]
        the_mi = MI[selected_linear[band][bin], 0]
        e1, e2 = map(lambda x: interesting_electrodes[band].index(x), selected_pairs[band][bin])
        tmp_vsmi = np.zeros((sel_el_num,sel_el_num))
        new_surr = np.zeros(99)
        for i in range(99):
            tmp_vsmi[np.triu_indices(sel_el_num, 1)] = VSMI[:,i]
            new_surr[i] = tmp_vsmi[e1,e2]
        plt.hist(old_surr, bins="auto", density=True, label="Surrogate")
        plt.hist(new_surr, bins="auto", density=True, label="Volt surr", alpha=0.8)
        plt.vlines(the_mi, 0, plt.ylim()[1],"r", 'solid', "True MI", lw=2)
        plt.title(band_names[band-1]+f" band $-$ r={bin/10-1:.2}")
        plt.legend()
        plt.show()

In [ ]:
[len(interesting_electrodes[band]) for band in interesting_electrodes], 9*17

In [ ]:
nle = NonLinearEstimator(bins=10, surrogates=0, cache="cache", workers=20, verbose=True)
data = loadmat(f"../NonLinearityResultsNew/NEW_electrodeBLP_fixedTime_avg1_surrogates/surrogateBLP_band1.mat")["BLP"]
nle.estimate(data, extended_stats=False, save_out=False, display=False)[1].shape

In [ ]:
plt.plot(np.arange(power.shape[0])/1000, power[:,0])
plt.plot(np.arange(dsp.shape[0])/8, dsp[:,0])

In [ ]:
dsp

In [ ]:
from scipy.io import loadmat
mat = loadmat("../NonLinearityData/EEG_el_so_BLP_NEW/NEW_electrodeBLP_fixedTime_avg1_band1.mat")
mat["EEG"].shape